# Total return swaps and variance swaps

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` and `02_pricing/pricing_across_asset_classes.ipynb`. **`trs_equity`** for index TRS vs a **financing leg**; **`variance_swap`** for realized vs strike variance with **`vega`-style** metrics when registered.


## Concept

- **TRS:** receive/pay **total return** (price + dividends) vs **funded financing** (curve + spread).
- **Variance swap:** payoff on realized variance vs **strike variance**; **vega notional** scales sensitivity.

Variance swap PV may be near zero without a full forward variance / surface stack—focus on the metric plumbing.


In [ ]:
from datetime import date
import math

from finstack_quant.core.dates import DayCount
from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext
from finstack_quant.core.money import Money
from finstack_quant.valuations.instruments import EquityTotalReturnSwap, VarianceSwap, price_instrument
from _shared import print_metrics, usd_sofr_fixings

AS_OF = date(2025, 1, 15)
MATURITY = date(2026, 1, 15)


## Contract terms and schedules


In [ ]:
underlying = {
    "ticker": "SPX", "currency": "USD", "spot_id": "SPX-SPOT", "div_yield_id": "SPX-DIV",
}
schedule = {
    "start": AS_OF, "end": MATURITY,
    "params": {
        "frequency": {"count": 3, "unit": "months"},
        "day_count": "act_360", "business_day_convention": "following",
        "calendar_id": "weekends_only", "end_of_month": False,
        "payment_lag_days": 0, "stub": "none",
    },
}
financing = {
    "discount_curve_id": "USD-OIS", "forward_curve_id": "USD-SOFR-3M",
    "spread_bp": "75", "day_count": "act_360",
}
trs = (
    EquityTotalReturnSwap.builder().id("TRS-SPX-1Y").side("receive")
    .notional(Money(5_000_000, "USD")).schedule(schedule)
    .underlying(underlying).financing_leg(financing)
    .dividend_settlement("on_dividend_date").dividend_tax_rate(0.0).build()
)
var_swap = (
    VarianceSwap.builder().id("VARSPX-1Y").side("receive")
    .notional(Money(1_000_000, "USD")).start_date(AS_OF).maturity(MATURITY)
    .strike_variance(0.04).underlying_ticker("SPX").spot_id("SPX-SPOT")
    .vol_surface_id("SPX-VOL").div_yield_id("SPX-DIV")
    .observation_frequency({"count": 1, "unit": "days"}).observation_calendar_id("USNY")
    .realized_var_method("close_to_close").price_series_policy("adjusted")
    .instrument_pricing_overrides({"market_quotes": {"implied_volatility": 0.20}})
    .day_count(DayCount.ACT_365F).discount_curve_id("USD-OIS").build()
)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85)],
    interp="monotone_convex",
    extrapolation="extend",
    day_count="act_365f",
)
# Projection factors P(t) = exp(-(0.05 t - 0.0005 t^2)): the instantaneous SOFR forward
# falls linearly from 5.00% today to 4.50% at five years.
sofr = ForwardCurve(
    {
        "id": "USD-SOFR-3M",
        "base_date": AS_OF,
        "knots": [(t, math.exp(-(0.05 * t - 0.0005 * t * t))) for t in (1.0, 2.0, 3.0, 4.0, 5.0)],
        "interp": "log_linear",
        "extrapolation": "extend",
        "day_count": "act_360",
        "index_tenor": "3M",
    }
)
mc = MarketContext().insert(ois).insert(sofr)
mc.insert_series(usd_sofr_fixings(AS_OF))
mc.insert_price("SPX-SPOT", 4500.0, "USD")
mc.insert_price("SPX-DIV", 0.015)
mc.insert_price("SPX_IMPL_VOL", 0.20)
print("market ready for TRS + variance")

## Pricing


In [ ]:
for label, instrument in (("trs", trs), ("variance", var_swap)):
    print(label, price_instrument(instrument, mc, AS_OF, model="discounting"))


## Metrics


In [ ]:
for label, instrument, mets in (
    ("trs", trs, ["financing_annuity", "index_delta", "par_spread"]),
    ("variance", var_swap, ["vega", "variance_expected", "variance_vega"]),
):
    out = price_instrument(instrument, mc, AS_OF, model="discounting", metrics=mets)
    print("—", label)
    print_metrics(out, mets)
print("Vega notional scales variance swap sensitivity to implied vol changes (see deal terms).")


## Takeaways

- **TRS** splits **asset leg** vs **financing leg**; curves must include `forward_curve_id` for float financing.
- **Variance swaps** need rich forward variance inputs for non-trivial PV; metrics still demonstrate the hook.
- Always **`validate_instrument_json`** after edits to long JSON specs.
